# 01 · Dataset setup

Loads the Kaggle *Wikipedia Movie Plots with Plot Summaries* dump (`data/wiki_movie_plots_deduped_with_summaries.csv`, 34 K rows), keeps the five fields the system needs (title, genre, plot, summary, release year), drops rows with missing text, normalises the text and removes duplicate titles.

**Output:** `data/cleaned_movie_plots.csv` (32,406 movies).

The second cleaning pass, `scripts/clean_dataset.py`, turns this file into `data/cleaned_movie_plots_v2.csv` (32,364 movies): it strips Wikipedia citation markers, deduplicates by canonical title + year, builds the composite `embedding_text` document and flags suspicious rows. Every later notebook reads the v2 file.

In [ ]:
import pandas as pd

df = pd.read_csv("../data/wiki_movie_plots_deduped_with_summaries.csv")

In [4]:
df = df[['Title', 'Genre', 'Plot', 'PlotSummary', 'Release Year']]
df = df.rename(columns={
    'Title': 'title',
    'Genre': 'genre',
    'Plot': 'plot',
    'PlotSummary': 'summary',
    'Release Year': 'release_year'
})
print(df.columns)

Index(['title', 'genre', 'plot', 'summary', 'release_year'], dtype='object')


In [6]:
df = df.dropna(subset=['title', 'genre', 'plot', 'summary', 'release_year'])

# Remove rows with empty strings or just spaces
df = df[
    (df['title'].str.strip() != '') &
    (df['genre'].str.strip() != '') &
    (df['plot'].str.strip() != '') &
    (df['summary'].str.strip() != '')
]

In [7]:
import re

def clean_text(text):
    text = text.lower()  # lowercase
    text = re.sub(r'\s+', ' ', text)  # remove extra spaces/newlines
    text = re.sub(r'[^a-zA-Z0-9\s.,!?\'"-]', '', text)  # remove weird symbols
    return text.strip()

df['plot'] = df['plot'].apply(clean_text)
df['summary'] = df['summary'].apply(clean_text)

In [8]:
df = df.drop_duplicates(subset=['title'])

In [9]:
df.to_csv("../data/cleaned_movie_plots.csv", index=False)
print("Data cleaned and saved to ../data/cleaned_movie_plots.csv")

Data cleaned and saved to cleaned_movie_plots.csv
